# Task 19 - Interaction Feature from Two Activity Channels (ratios, engagement index, polynomial terms)

**Goal:** combine two activity measures into interaction-based metrics (transition ratio, engagement index), detect **non-linear** relationships with polynomial features, and optimise the feature set for predictive performance.

**Data mapping (please read):** your data has no `time on app` or `time on website`. The method does not depend on the column names, so two channel columns are configurable:

| Task wording | Used here (change `CHANNEL_A` / `CHANNEL_B` to remap) |
|---|---|
| time on app | `CHANNEL_A = "rating_count"` - feedback activity |
| time on website | `CHANNEL_B = "units_sold"` - purchase activity |
| app-to-web transition ratio | ratings per unit sold (feedback conversion) |
| optimisation target | product `rating` (reliably rated products only) |

If you obtain real app/web time columns, set the two names and rerun; nothing else changes.
**Assumption:** your message was cut off after "optimizing it fo...". I assumed *optimising for predictive performance*; tell me if you meant something else.

**Input:** newest file before this task. **Outputs:** `data/processed/womens_clothing_channel_interaction_features.csv`, `reports/task_19_interaction_feature_comparison.csv`, `reports/task_19_nonlinearity_tests.csv`, `reports/task_19_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import mutual_info_regression
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
CHANNEL_A, CHANNEL_B = "rating_count", "units_sold"
TARGET_COL = "rating"
MIN_RATINGS = 20

## 1. Load data

In [ ]:
df, src = load_first(*[PROC / n for n in FULL[3:]])
a_col, b_col = pick(df, CHANNEL_A), pick(df, CHANNEL_B)
t_col = pick(df, TARGET_COL); rc_col = pick(df, "rating_count")
for c in (a_col, b_col): assert df[c].min() >= 0, f"{c} looks scaled - use an un-scaled file"
mask = df[rc_col] >= MIN_RATINGS
print(f"Products: {len(df):,} | used for modelling (rating_count >= {MIN_RATINGS}): {mask.sum():,}")
df[[a_col, b_col, t_col]].describe().round(2)

## 2. Interaction-based metrics

| Feature | Meaning |
|---|---|
| `ch_log_a`, `ch_log_b` | log1p of each channel (both are heavily right-skewed) |
| `ch_a_to_b_ratio` | A per unit of B (winsorised at the 99th percentile) |
| `ch_a_share` | A / (A + B): share of activity in channel A |
| `ch_log_ratio` | log((A+1)/(B+1)): symmetric version of the transition ratio |
| `ch_total_log` | log1p(A + B): overall activity |
| `ch_balance` | 1 - |A - B| / (A + B): 1 = perfectly balanced use of both channels |
| `ch_geo_engagement` | geometric mean sqrt(A x B) in log space: high only if *both* channels are active |
| `ch_engagement_index` | weighted blend of standardised log A, log B and balance |

In [ ]:
A, B = df[a_col].astype(float), df[b_col].astype(float)
ch = pd.DataFrame(index=df.index)
ch["ch_log_a"], ch["ch_log_b"] = np.log1p(A), np.log1p(B)
ratio = (A / B.replace(0, np.nan)); ch["ch_a_to_b_ratio"] = ratio.clip(upper=ratio.quantile(.99)).fillna(0)
ch["ch_a_share"] = (A / (A + B).replace(0, np.nan)).fillna(0.5)
ch["ch_log_ratio"] = np.log((A + 1) / (B + 1))
ch["ch_total_log"] = np.log1p(A + B)
ch["ch_balance"] = (1 - (A - B).abs() / (A + B).replace(0, np.nan)).fillna(0)
ch["ch_geo_engagement"] = 0.5 * (ch["ch_log_a"] + ch["ch_log_b"])
zs = lambda s: (s - s.mean()) / (s.std() + 1e-9)
ch["ch_engagement_index"] = 0.4 * zs(ch["ch_log_a"]) + 0.4 * zs(ch["ch_log_b"]) + 0.2 * zs(ch["ch_balance"])
ch.describe().T.round(3)

## 3. Detect non-linear relationships with the target

In [ ]:
d = pd.concat([ch, df[t_col].rename("target")], axis=1)[mask].dropna()
rows = []
kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
for c in ch.columns:
    x = d[[c]].values; y = d["target"].values
    pear = stats.pearsonr(d[c], y)[0]; spear = stats.spearmanr(d[c], y)[0]
    mi = mutual_info_regression(x, y, random_state=RANDOM_STATE)[0]
    lin = cross_val_score(LinearRegression(), x, y, cv=kf, scoring="r2").mean()
    quad = cross_val_score(make_pipeline(StandardScaler(), PolynomialFeatures(2), LinearRegression()), x, y, cv=kf, scoring="r2").mean()
    cub = cross_val_score(make_pipeline(StandardScaler(), PolynomialFeatures(3), LinearRegression()), x, y, cv=kf, scoring="r2").mean()
    rows.append({"feature": c, "pearson": pear, "spearman": spear, "abs_spearman_minus_pearson": abs(spear) - abs(pear),
                 "mutual_info": mi, "cv_r2_linear": lin, "cv_r2_quadratic": quad, "cv_r2_cubic": cub,
                 "nonlinear_gain": max(quad, cub) - lin})
nl = pd.DataFrame(rows).sort_values("nonlinear_gain", ascending=False).reset_index(drop=True)
nl["nonlinear_flag"] = (nl["nonlinear_gain"] > 0.005) | (nl["abs_spearman_minus_pearson"] > 0.05)
nl.round(4)

In [ ]:
# Decile curves: a flat line means no relationship, a curve means non-linearity
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a_, c in zip(ax, ["ch_log_a", "ch_log_b", "ch_log_ratio"]):
    q = pd.qcut(d[c].rank(method="first"), 10, labels=False)
    g = d.groupby(q).agg(x=(c, "mean"), y=("target", "mean"), se=("target", lambda s: s.std() / np.sqrt(len(s))))
    a_.errorbar(g["x"], g["y"], yerr=g["se"], marker="o"); a_.set_title(f"{TARGET_COL} by decile of {c}"); a_.set_xlabel(c)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=d, x="ch_log_a", y="ch_log_b", hue="target", palette="viridis", alpha=.6, ax=ax[0]); ax[0].set_title("Channel A vs B, coloured by target")
hm = d.pivot_table(index=pd.qcut(d["ch_log_b"], 5, duplicates="drop"), columns=pd.qcut(d["ch_log_a"], 5, duplicates="drop"), values="target", aggfunc="mean")
sns.heatmap(hm, annot=True, fmt=".2f", cmap="RdYlGn", ax=ax[1]); ax[1].set_title("Mean target: B quintile (rows) x A quintile (cols)")
plt.tight_layout(); plt.show()

## 4. Polynomial feature generation and optimisation

Search over base feature sets, polynomial degree (1-3), with / without pure powers, and regularisation strength (RidgeCV picks alpha inside each fold). The simplest configuration within 1 standard error of the best CV R-squared wins.

In [ ]:
base_sets = {"A,B (logs)": ["ch_log_a", "ch_log_b"],
             "A,B + ratio": ["ch_log_a", "ch_log_b", "ch_log_ratio"],
             "A,B + ratio + balance": ["ch_log_a", "ch_log_b", "ch_log_ratio", "ch_balance"],
             "all interaction metrics": list(ch.columns)}
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE); y = d["target"].values; rows = []
for bname, cols in base_sets.items():
    for deg in (1, 2, 3):
        for inter_only in ((False, True) if deg > 1 else (False,)):
            if bname == "all interaction metrics" and deg == 3: continue          # too many terms for ~1k rows
            pipe = make_pipeline(StandardScaler(), PolynomialFeatures(deg, interaction_only=inter_only, include_bias=False),
                                 StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
            sc = cross_val_score(pipe, d[cols].values, y, cv=cv, scoring="r2")
            n_terms = PolynomialFeatures(deg, interaction_only=inter_only, include_bias=False).fit(d[cols].values).n_output_features_
            rows.append({"base_set": bname, "degree": deg, "interaction_only": inter_only, "n_terms": n_terms,
                         "cv_r2_mean": sc.mean(), "cv_r2_se": sc.std() / np.sqrt(len(sc))})
grid = pd.DataFrame(rows).sort_values("cv_r2_mean", ascending=False).reset_index(drop=True)
best = grid.iloc[0]
simple = grid[grid["cv_r2_mean"] >= best["cv_r2_mean"] - best["cv_r2_se"]].sort_values(["n_terms", "degree"]).iloc[0]
print("Best CV R2:", round(best["cv_r2_mean"], 4), "| chosen (simplest within 1 SE):", dict(simple[["base_set", "degree", "interaction_only", "n_terms"]]))
grid.round(4)

In [ ]:
# Is anything beyond a linear baseline worth keeping?
lin_base = grid[(grid["degree"] == 1)].sort_values("cv_r2_mean", ascending=False).iloc[0]
print(f"Linear baseline CV R2 = {lin_base['cv_r2_mean']:.4f} | chosen polynomial CV R2 = {simple['cv_r2_mean']:.4f} | gain = {simple['cv_r2_mean'] - lin_base['cv_r2_mean']:+.4f}")
plt.figure(figsize=(9, 4)); g = grid.assign(cfg=grid["base_set"].str[:12] + " d" + grid["degree"].astype(str) + np.where(grid["interaction_only"], "i", ""))
sns.barplot(data=g.head(12), x="cv_r2_mean", y="cfg", color="#4c72b0"); plt.title("Top configurations (CV R2)"); plt.tight_layout(); plt.show()

## 5. Select the polynomial terms to keep

Lasso (CV) on the chosen polynomial expansion keeps only terms with non-zero weight; those terms become dataset columns (`ch_poly_*`). Terms are computed from *standardised* inputs, using statistics fitted on all products.

In [ ]:
cols = base_sets[simple["base_set"]]
scaler = StandardScaler().fit(ch[cols])
pf = PolynomialFeatures(int(simple["degree"]), interaction_only=bool(simple["interaction_only"]), include_bias=False).fit(scaler.transform(ch[cols]))
P_all = pd.DataFrame(pf.transform(scaler.transform(ch[cols])), index=df.index,
                     columns=[re.sub(r"\s+", "*", n.replace("^", "_pow")) for n in pf.get_feature_names_out(cols)])
P_fit = P_all.loc[d.index]
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20000)).fit(P_fit, d["target"])
coef = pd.Series(lasso[-1].coef_, index=P_fit.columns)
kept = coef[coef != 0].abs().sort_values(ascending=False)
if kept.empty:
    print("Lasso zeroed every term -> the interactions carry no usable signal for this target; keeping the 5 most informative terms by mutual information.")
    mi = pd.Series(mutual_info_regression(P_fit, d["target"], random_state=RANDOM_STATE), index=P_fit.columns); kept = mi.sort_values(ascending=False).head(5)
kept_terms = list(kept.index[:12])
print("Kept terms:", kept_terms)
poly_out = P_all[kept_terms].add_prefix("ch_poly_")
coef.loc[kept_terms].round(4)

## 6. Final check and save

In [ ]:
eng = ["ch_engagement_index", "ch_geo_engagement", "ch_log_ratio", "ch_balance"]
results = []
for name, X_ in {"A,B logs only (baseline)": ch[["ch_log_a", "ch_log_b"]], "interaction metrics": ch[eng + ["ch_log_a", "ch_log_b"]],
                 "selected polynomial terms": poly_out, "interaction + polynomial": pd.concat([ch[eng], poly_out], axis=1)}.items():
    sc = cross_val_score(make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))), X_.loc[d.index].values, y, cv=cv, scoring="r2")
    results.append({"feature_set": name, "n_features": X_.shape[1], "cv_r2_mean": sc.mean(), "cv_r2_std": sc.std()})
final_cmp = pd.DataFrame(results).round(4); final_cmp

In [ ]:
out = pd.concat([df, ch, poly_out], axis=1)
out_path = PROC / "womens_clothing_channel_interaction_features.csv"; out.to_csv(out_path, index=False)
grid.assign(selected=(grid.index == grid[(grid["base_set"] == simple["base_set"]) & (grid["degree"] == simple["degree"]) & (grid["interaction_only"] == simple["interaction_only"])].index[0])).round(5).to_csv(REPORTS / "task_19_interaction_feature_comparison.csv", index=False)
nl.round(5).to_csv(REPORTS / "task_19_nonlinearity_tests.csv", index=False)
base_cols = pd.read_csv(src, nrows=1).columns
status = pd.DataFrame([{"task": "Task 19 - Channel Interaction Features", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": src.name, "rows": len(out), "channel_a": a_col, "channel_b": b_col, "target": t_col, "rows_modelled": int(mask.sum()),
    "chosen_base_set": simple["base_set"], "chosen_degree": int(simple["degree"]), "linear_cv_r2": round(lin_base["cv_r2_mean"], 4),
    "chosen_cv_r2": round(simple["cv_r2_mean"], 4), "n_nonlinear_features": int(nl["nonlinear_flag"].sum()),
    "columns_added": len([c for c in out.columns if c not in base_cols]), "output_dataset": out_path.name}])
status.to_csv(REPORTS / "task_19_analysis_status.csv", index=False); status.T

## 7. Key takeaways (fill in after running)

- Which interaction metrics show non-linear behaviour (flags in section 3) and what shape do the decile curves have?
- Does the polynomial model beat the linear baseline by a meaningful margin? If the gain is near zero, say so: the engineered features then add interpretability, not accuracy.
- Product ratings are averages over many buyers, so a low R-squared is expected; judge the features by *relative* improvement.
- `ch_*` columns are built from `rating_count` and `units_sold`; avoid using them to predict those same columns.